### Import Dependencies

In [2]:
import re
from typing import Any, Dict

from guardrails import Guard, OnFailAction

from guardrails.validator_base import (
    FailResult,
    PassResult,
    ValidationResult,
    Validator,
    register_validator
)

### Product ID detection Guardrail

In [3]:
PRODUCT_ID = re.compile(r"\bB0[A-Z0-9]{8}\b")

In [4]:
PRODUCT_ID.findall("dasdasd B0B87HSZ8K asdfasfgadsf B0BVRLW1DP")

['B0B87HSZ8K', 'B0BVRLW1DP']

In [5]:
@register_validator(name="detect_product_id", data_type="string")
class DetectProductID(Validator):

    def _validate(self, value: Any, metadata: Dict[str, Any] = {}) -> ValidationResult:

        matches = PRODUCT_ID.findall(value)

        if matches:
            return FailResult(
                error_message=f"found {len(matches)} product ids: {matches}",
            )

        return PassResult()

In [6]:
guard = Guard().use(DetectProductID(on_fail=OnFailAction.NOOP))

In [7]:
text = "Added 5 units of OYEEICE iPad Air 5th/4th Generation Leather Case (product ID: B09LXP9Z83) to your shopping cart. And one more B09X9RM6FD"

In [8]:
guardrail_outcome = guard.validate(text)

/Users/hang/Documents/Development/ai-eng-course/ai-eng-project/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [ ]:
guardrail_outcome

ValidationOutcome[TypeVar](call_id='4790813312', raw_llm_output='Added 5 units of OYEEICE iPad Air 5th/4th Generation Leather Case (product ID: B09LXP9Z83) to your shopping cart. And one more B09X9RM6FD', validation_summaries=[ValidationSummary(validator_name='DetectProductID', validator_status='fail', property_path='$', failure_reason="found 2 product ids: ['B09LXP9Z83', 'B09X9RM6FD']", error_spans=None)], validated_output='Added 5 units of OYEEICE iPad Air 5th/4th Generation Leather Case (product ID: B09LXP9Z83) to your shopping cart. And one more B09X9RM6FD', reask=None, validation_passed=False, error=None)

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [10]:
for log in guard.history.last.failed_validations:
    print("Exception: ", log.validation_result.error_message)

Exception:  found 2 product ids: ['B09LXP9Z83', 'B09X9RM6FD']


In [11]:
clean_text = "Some random text"

In [12]:
guardrail_outcome_clean = guard.validate(clean_text)

In [ ]:
guardrail_outcome_clean

ValidationOutcome[TypeVar](call_id='4790828512', raw_llm_output='Some random text', validation_summaries=[], validated_output='Some random text', reask=None, validation_passed=True, error=None)

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


### Product ID Redaction Guardrail

In [14]:
PRODUCT_ID = re.compile(r"\bB0[A-Z0-9]{8}\b")
REPLACEMENT = "[REDACTED..]"

In [15]:
PRODUCT_ID.sub(REPLACEMENT, "dasdasd B0B87HSZ8K asdfasfgadsf B0BVRLW1DP")

'dasdasd [REDACTED..] asdfasfgadsf [REDACTED..]'

In [16]:
@register_validator(name="redact_product_id", data_type="string")
class RedactProductID(Validator):

    def _validate(self, value: Any, metadata: Dict[str, Any] = {}) -> ValidationResult:

        matches = PRODUCT_ID.findall(value)

        if matches:
            return FailResult(
                error_message=f"found {len(matches)} product ids: {matches}",
                fix_value=PRODUCT_ID.sub(REPLACEMENT, value)
            )

        return PassResult()

In [17]:
guard_redact = Guard().use(RedactProductID(on_fail=OnFailAction.FIX))

In [18]:
outcome_redact = guard_redact.validate(text)

In [19]:
outcome_redact

ValidationOutcome[TypeVar](call_id='4790982272', raw_llm_output='Added 5 units of OYEEICE iPad Air 5th/4th Generation Leather Case (product ID: B09LXP9Z83) to your shopping cart. And one more B09X9RM6FD', validation_summaries=[ValidationSummary(validator_name='RedactProductID', validator_status='fail', property_path='$', failure_reason="found 2 product ids: ['B09LXP9Z83', 'B09X9RM6FD']", error_spans=None)], validated_output='Added 5 units of OYEEICE iPad Air 5th/4th Generation Leather Case (product ID: [REDACTED..]) to your shopping cart. And one more [REDACTED..]', reask=None, validation_passed=True, error=None)

In [20]:
print(f"Output: {outcome_redact.validated_output}")

Output: Added 5 units of OYEEICE iPad Air 5th/4th Generation Leather Case (product ID: [REDACTED..]) to your shopping cart. And one more [REDACTED..]


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.
